# Multi-Agent Governance — Dev Log

## Objetivo

`core/multi_agent_governance` é a Onda 4 do V2 — a mais complexa
arquiteturalmente. Este módulo registra declarativamente (`agents.yaml`)
quais agentes lógicos existem no AthenaGov AI e quais ações cada um pode
executar, com `authorize(agent_id, action)` como ponto central de checagem.

**Escopo honesto**: é uma checagem declarativa explícita (mesmo estilo do
`policy_engine`), não um interceptor real de chamadas Python via reflexão —
quem orquestra decide chamar `authorize()` e o que fazer com o resultado.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.multi_agent_governance.registry import authorize, list_agents

agents = list_agents()
print(f"{len(agents)} agente(s) registrados:")
for a in agents:
    print(f"  - {a.agent_id} ({a.name}) | risco={a.risk_tier.value} | {len(a.allowed_actions)} ação(ões)")
print()
r1 = authorize("red_teamer", "red_team_lab.run_suite")
r2 = authorize("red_teamer", "policy_engine.evaluate")
print(f"authorize('red_teamer', 'red_team_lab.run_suite') -> {r1.authorized} | {r1.reason}")
print(f"authorize('red_teamer', 'policy_engine.evaluate') -> {r2.authorized} | {r2.reason}")

5 agente(s) registrados:
  - ripd_generator (Gerador de RIPD) | risco=medium | 7 ação(ões)
  - auditor (Agente de Auditoria) | risco=low | 5 ação(ões)
  - red_teamer (Agente de Red Team) | risco=high | 2 ação(ões)
  - reviewer (Agente de Revisão Humana (interface)) | risco=low | 3 ação(ões)
  - sandbox_explorer (Agente de Simulação) | risco=low | 2 ação(ões)

authorize('red_teamer', 'red_team_lab.run_suite') -> True | Ação 'red_team_lab.run_suite' está na lista de ações permitidas do agente 'Agente de Red Team'.
authorize('red_teamer', 'policy_engine.evaluate') -> False | Ação 'policy_engine.evaluate' NÃO está na lista de ações permitidas do agente 'Agente de Red Team'.


## Testes e Handoff

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/multi_agent_governance/tests -v
```

7/7 testes passando. **TODO onda futura**: integração automática via
decorator amarrando `authorize()` a chamadas reais de módulo.